# Автоматическая генерация признаков FAR-Trans

## Шаг 1. План признаков

В ручном baseline мы сами выбрали пять признаков. Здесь программа будет создавать варианты по заданным правилам. Человек задаёт рамки поиска; программа перебирает комбинации внутри них.

Начинаем с числа записанных покупок `Buy`:

| Группа | Что будем считать |
|---|---|
| `user` | Все покупки клиента |
| `item` | Покупки актива всеми клиентами |
| `user_item` | Покупки конкретного актива этим клиентом |

Для каждой группы берём окна **7, 14, 30 и 60 дней до даты прогноза**. Все окна исключают события в день прогноза и после него. Это первая ограниченная сетка поиска, а не заранее доказанный лучший набор.

Например, `user_item_buy_count_14d` — сколько раз клиент покупал этот актив за предыдущие 14 дней.

**Сейчас создаём только описания 12 вариантов. Сделки ещё не загружаем и признаки по ним не считаем.**


### Запусти следующую ячейку: Shift + Enter

- Первый `for` выбирает группу: клиент, актив или пара клиент–актив.
- Второй `for` перебирает четыре периода для этой группы.
- `append()` добавляет описание комбинации в список `feature_plan`.
- `f"..."` вставляет группу и период в название признака.

Три группы × четыре периода = **12 вариантов**.


In [1]:
groups = ["user", "item", "user_item"]
windows = [7, 14, 30, 60]

feature_plan = []

for group in groups:
    for days in windows:
        feature_plan.append({
            "name": f"{group}_buy_count_{days}d",
            "group": group,
            "days": days
        })

print("Количество вариантов:", len(feature_plan))

for feature in feature_plan:
    print(feature)


Количество вариантов: 12
{'name': 'user_buy_count_7d', 'group': 'user', 'days': 7}
{'name': 'user_buy_count_14d', 'group': 'user', 'days': 14}
{'name': 'user_buy_count_30d', 'group': 'user', 'days': 30}
{'name': 'user_buy_count_60d', 'group': 'user', 'days': 60}
{'name': 'item_buy_count_7d', 'group': 'item', 'days': 7}
{'name': 'item_buy_count_14d', 'group': 'item', 'days': 14}
{'name': 'item_buy_count_30d', 'group': 'item', 'days': 30}
{'name': 'item_buy_count_60d', 'group': 'item', 'days': 60}
{'name': 'user_item_buy_count_7d', 'group': 'user_item', 'days': 7}
{'name': 'user_item_buy_count_14d', 'group': 'user_item', 'days': 14}
{'name': 'user_item_buy_count_30d', 'group': 'user_item', 'days': 30}
{'name': 'user_item_buy_count_60d', 'group': 'user_item', 'days': 60}


In [3]:
from pathlib import Path
import pandas as pd

project_dir = Path.cwd()
if project_dir.name == "outputs":
    project_dir = project_dir.parent

transactions = pd.read_csv(
    project_dir / "work/data/FAR-Trans/transactions.csv",
    dtype={"customerID": "string", "ISIN": "string"}
)

transactions["timestamp"] = pd.to_datetime(
    transactions["timestamp"]
)

prediction_date = pd.Timestamp("2020-04-01")

past_buys = transactions[
    (transactions["transactionType"] == "Buy")
    & (transactions["timestamp"] >= prediction_date - pd.Timedelta(days=60))
    & (transactions["timestamp"] < prediction_date)
].copy()

del transactions

print("Покупок за предыдущие 60 дней:", len(past_buys))
past_buys[["customerID", "ISIN", "timestamp"]].head()

Покупок за предыдущие 60 дней: 18219


,customerID,ISIN,timestamp
0,00017496858921195E5A,GRS434003000,2020-03-27
17,001270E70A4ECA1CD664,GRS434003000,2020-03-09
45,0149804312DBC4F1A6F0,GRS434003000,2020-03-10
46,0149804312DBC4F1A6F0,GRS434003000,2020-03-10
52,017B4C60E3DE9D9986CF,GRS434003000,2020-03-12


In [4]:
# Берём клиента из первой показанной строки
client_id = past_buys.iloc[0]["customerID"]

# Берём первое описание из нашего плана
feature = feature_plan[0]

window_start = prediction_date - pd.Timedelta(
    days=feature["days"]
)

client_buys = past_buys[
    (past_buys["customerID"] == client_id)
    & (past_buys["timestamp"] >= window_start)
]

value = len(client_buys)

print("Клиент:", client_id)
print("Признак:", feature["name"])
print("Значение:", value)

client_buys[["ISIN", "timestamp"]]

Клиент: 00017496858921195E5A
Признак: user_buy_count_7d
Значение: 1


,ISIN,timestamp
0,GRS434003000,2020-03-27


In [5]:
import json

# Восстанавливаем прежний состав клиентов и активов
baseline = json.loads(
    (project_dir / "outputs/manual_baseline_reviewed_results.json")
    .read_text(encoding="utf-8")
)

candidate_pairs = pd.DataFrame({
    "customerID": baseline["client_ids"]
}).merge(
    pd.DataFrame({"ISIN": baseline["asset_ids"]}),
    how="cross"
)

# Какие столбцы определяют каждую группу
group_columns = {
    "user": ["customerID"],
    "item": ["ISIN"],
    "user_item": ["customerID", "ISIN"]
}


def generate_features(history, pairs, cutoff, plan):
    result = pairs.copy()

    for feature in plan:
        start = cutoff - pd.Timedelta(days=feature["days"])

        recent_buys = history[
            (history["timestamp"] >= start)
            & (history["timestamp"] < cutoff)
        ]

        columns = group_columns[feature["group"]]
        name = feature["name"]

        counts = (
            recent_buys.groupby(columns)
            .size()
            .reset_index(name=name)
        )

        result = result.merge(
            counts,
            on=columns,
            how="left",
            validate="many_to_one"
        )

        result[name] = result[name].fillna(0).astype(int)

    result["prediction_date"] = cutoff
    return result


auto_features = generate_features(
    past_buys,
    candidate_pairs,
    prediction_date,
    feature_plan
)

assert len(auto_features) == 500 * 50
assert not auto_features.duplicated(["customerID", "ISIN"]).any()

print("Пар клиент–актив:", len(auto_features))
print("Признаков:", len(feature_plan))

auto_features.head()

Пар клиент–актив: 25000
Признаков: 12


,customerID,ISIN,user_buy_count_7d,user_buy_count_14d,user_buy_count_30d,user_buy_count_60d,item_buy_count_7d,item_buy_count_14d,item_buy_count_30d,item_buy_count_60d,user_item_buy_count_7d,user_item_buy_count_14d,user_item_buy_count_30d,user_item_buy_count_60d,prediction_date
0,485EF2CE5FB9D9A7A08C,GRS003003035,1,3,7,13,260,772,2034,2706,1,3,7,13,2020-04-01
1,485EF2CE5FB9D9A7A08C,GRS015003007,1,3,7,13,222,555,1134,1524,0,0,0,0,2020-04-01
2,485EF2CE5FB9D9A7A08C,GRS323003012,1,3,7,13,144,347,893,1402,0,0,0,0,2020-04-01
3,485EF2CE5FB9D9A7A08C,GRS393503008,1,3,7,13,70,188,523,989,0,0,0,0,2020-04-01
4,485EF2CE5FB9D9A7A08C,GRS014003032,1,3,7,13,98,237,652,954,0,0,0,0,2020-04-01


In [6]:
# Загружаем историю для всех месяцев train и validation
transactions = pd.read_csv(
    project_dir / "work/data/FAR-Trans/transactions.csv",
    dtype={"customerID": "string", "ISIN": "string"}
)
transactions["timestamp"] = pd.to_datetime(transactions["timestamp"])

test_start = pd.Timestamp("2022-07-01")

all_buys = transactions[
    (transactions["transactionType"] == "Buy")
    & (transactions["timestamp"] >= "2020-01-01")
    & (transactions["timestamp"] < test_start)
].copy()

del transactions

manual_features = baseline["features"]
auto_names = [feature["name"] for feature in feature_plan]
feature_names = manual_features + auto_names
pair_keys = ["customerID", "ISIN"]


def make_dataset_month(date):
    date = pd.Timestamp(date)
    end = date + pd.DateOffset(months=1)

    if not date.is_month_start or date != date.normalize():
        raise ValueError("Нужно первое число месяца")

    if date < pd.Timestamp("2020-04-01") or end > test_start:
        raise ValueError("Строим только месяцы train/validation")

    if end > pd.Timestamp(baseline["observation"]["end_exclusive"]):
        raise ValueError("Месяц выходит за документированный период")

    # Считаем 12 новых признаков нашей готовой функцией
    result = generate_features(
        all_buys, candidate_pairs, date, feature_plan
    )

    # Добавляем пять прежних признаков
    for months in [1, 3]:
        past = all_buys[
            (all_buys["timestamp"] >= date - pd.DateOffset(months=months))
            & (all_buys["timestamp"] < date)
        ]

        groups_to_count = (
            ["user", "user_item", "item"]
            if months == 1
            else ["user", "user_item"]
        )

        for group in groups_to_count:
            columns = group_columns[group]
            name = f"{group}_buy_count_{months}m"

            counts = (
                past.groupby(columns)
                .size()
                .reset_index(name=name)
            )

            result = result.merge(
                counts, on=columns,
                how="left", validate="many_to_one"
            )
            result[name] = result[name].fillna(0).astype(int)

    # Покупки целевого месяца используются только для ответа
    future = all_buys[
        (all_buys["timestamp"] >= date)
        & (all_buys["timestamp"] < end)
    ]

    labels = future[pair_keys].drop_duplicates().assign(label=1)

    result = result.merge(
        labels, on=pair_keys,
        how="left", validate="one_to_one"
    )
    result["label"] = result["label"].fillna(0).astype(int)

    return result[
        pair_keys + ["prediction_date"] + feature_names + ["label"]
    ]


# Собираем месяцы в общую таблицу
dates = pd.date_range("2020-04-01", "2022-06-01", freq="MS")

dataset = pd.concat(
    [make_dataset_month(date) for date in dates],
    ignore_index=True
)

train = dataset[dataset["prediction_date"] < "2022-01-01"].copy()
validation = dataset[dataset["prediction_date"] >= "2022-01-01"].copy()

# Проверяем уникальность строк, пропуски и ответы
assert not dataset.duplicated(pair_keys + ["prediction_date"]).any()
assert not dataset[feature_names].isna().any().any()
assert dataset["label"].isin([0, 1]).all()
assert dataset.groupby(["customerID", "prediction_date"]).size().eq(50).all()

print("Train:", len(train), "| Validation:", len(validation))
print("Признаков:", len(feature_names), "= 5 ручных + 12 новых")

train.head()

Train: 525000 | Validation: 150000
Признаков: 17 = 5 ручных + 12 новых


,customerID,ISIN,prediction_date,user_buy_count_1m,user_item_buy_count_1m,item_buy_count_1m,user_buy_count_3m,user_item_buy_count_3m,user_buy_count_7d,user_buy_count_14d,...,user_buy_count_60d,item_buy_count_7d,item_buy_count_14d,item_buy_count_30d,item_buy_count_60d,user_item_buy_count_7d,user_item_buy_count_14d,user_item_buy_count_30d,user_item_buy_count_60d,label
0,485EF2CE5FB9D9A7A08C,GRS003003035,2020-04-01,7,7,2034,14,14,1,3,...,13,260,772,2034,2706,1,3,7,13,1
1,485EF2CE5FB9D9A7A08C,GRS015003007,2020-04-01,7,0,1134,14,0,1,3,...,13,222,555,1134,1524,0,0,0,0,0
2,485EF2CE5FB9D9A7A08C,GRS323003012,2020-04-01,7,0,893,14,0,1,3,...,13,144,347,893,1402,0,0,0,0,0
3,485EF2CE5FB9D9A7A08C,GRS393503008,2020-04-01,7,0,523,14,0,1,3,...,13,70,188,523,989,0,0,0,0,0
4,485EF2CE5FB9D9A7A08C,GRS014003032,2020-04-01,7,0,652,14,0,1,3,...,13,98,237,652,954,0,0,0,0,0


In [7]:
import os
import hashlib
import platform

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

import joblib
import numpy as np
import sklearn

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import log_loss, roc_auc_score
from threadpoolctl import threadpool_limits

seed = 42

inner_train = train[
    train["prediction_date"] < "2021-07-01"
].copy()

inner_check = train[
    train["prediction_date"] >= "2021-07-01"
].copy()

assert (
    inner_train["prediction_date"].max()
    + pd.DateOffset(months=1)
    <= inner_check["prediction_date"].min()
)

print("Обучение для отбора:", len(inner_train))
print("Проверка для отбора:", len(inner_check))

Обучение для отбора: 375000
Проверка для отбора: 150000


In [8]:
max_new_features = 5
permutation_repeats = 3

selector_model = HistGradientBoostingClassifier(
    max_iter=100,
    max_leaf_nodes=7,
    early_stopping=False,
    random_state=seed
)

with threadpool_limits(limits=4):
    selector_model.fit(
        inner_train[feature_names],
        inner_train["label"]
    )

    inner_probabilities = selector_model.predict_proba(
        inner_check[feature_names]
    )[:, 1]

    importance = permutation_importance(
        selector_model,
        inner_check[feature_names],
        inner_check["label"],
        scoring="neg_log_loss",
        n_repeats=permutation_repeats,
        random_state=seed,
        n_jobs=1
    )

print(
    "LogLoss вспомогательной модели:",
    log_loss(inner_check["label"], inner_probabilities)
)

importance_table = pd.DataFrame({
    "feature": feature_names,
    "importance": importance.importances_mean,
    "shuffle_std": importance.importances_std
})

new_importance = importance_table[
    importance_table["feature"].isin(auto_names)
].sort_values(
    ["importance", "feature"],
    ascending=[False, True]
)

selected_new = (
    new_importance[new_importance["importance"] > 0]
    .head(max_new_features)["feature"]
    .tolist()
)

selected_features = manual_features + selected_new

print(new_importance.round(6).to_string(index=False))
print("\nВыбраны новые признаки:", selected_new)
print("Всего признаков:", len(selected_features))

LogLoss вспомогательной модели: 0.02595600448507737
                feature  importance  shuffle_std
     item_buy_count_60d    0.000527     0.000061
user_item_buy_count_60d    0.000494     0.000008
     item_buy_count_14d    0.000388     0.000016
     user_buy_count_60d    0.000297     0.000009
user_item_buy_count_14d    0.000128     0.000002
      item_buy_count_7d    0.000104     0.000015
      user_buy_count_7d    0.000081     0.000013
     user_buy_count_30d    0.000056     0.000003
     item_buy_count_30d    0.000017     0.000008
user_item_buy_count_30d    0.000004     0.000001
 user_item_buy_count_7d   -0.000004     0.000000
     user_buy_count_14d   -0.000028     0.000003

Выбраны новые признаки: ['item_buy_count_60d', 'user_item_buy_count_60d', 'item_buy_count_14d', 'user_buy_count_60d', 'user_item_buy_count_14d']
Всего признаков: 10


In [9]:
case_keys = ["customerID", "prediction_date"]


def tie_key(customer, date, asset):
    payload = json.dumps(
        [seed, str(customer), str(pd.Timestamp(date).date()), str(asset)],
        ensure_ascii=False,
        separators=(",", ":")
    ).encode("utf-8")

    return int.from_bytes(
        hashlib.blake2b(
            payload,
            digest_size=8,
            person=b"FAR-tie"
        ).digest(),
        "big"
    )


validation = validation.copy()
validation["_tie"] = np.array([
    tie_key(customer, date, asset)
    for customer, date, asset in validation[
        ["customerID", "prediction_date", "ISIN"]
    ].itertuples(index=False, name=None)
], dtype=np.uint64)


def top_five(data, probabilities=None, criteria=None):
    ranked = data.copy()

    if probabilities is not None:
        scores = (
            probabilities
            if isinstance(probabilities, pd.Series)
            else pd.Series(probabilities, index=data.index)
        )
        ranked["_score"] = scores.reindex(data.index)
        assert ranked["_score"].notna().all()
        criteria = ["_score"]

    ranked = ranked.sort_values(
        case_keys + criteria + ["_tie", "ISIN"],
        ascending=[True, True] + [False] * len(criteria) + [True, True]
    )

    top = ranked.groupby(case_keys, sort=False).head(5).copy()
    top["_rank"] = top.groupby(case_keys, sort=False).cumcount() + 1
    return top


def case_scores(data, top):
    cases = data.groupby(case_keys)["label"].sum().rename(
        "positive_assets"
    ).to_frame()

    cases["hits"] = top.groupby(case_keys)["label"].sum().reindex(
        cases.index, fill_value=0
    )

    eligible = cases["positive_assets"] > 0

    cases["Recall@5"] = (
        cases["hits"] / cases["positive_assets"]
    ).where(eligible)

    cases["Precision@5"] = (cases["hits"] / 5).where(eligible)
    cases["Precision@5_all"] = cases["hits"] / 5

    gains = top.assign(
        gain=top["label"] / np.log2(top["_rank"] + 1)
    )
    dcg = gains.groupby(case_keys)["gain"].sum().reindex(
        cases.index, fill_value=0
    )

    ideal = np.r_[0., np.cumsum(1 / np.log2(np.arange(1, 6) + 1))]
    idcg = ideal[
        cases["positive_assets"].clip(upper=5).to_numpy(dtype=int)
    ]

    cases["NDCG@5"] = np.divide(
        dcg, idcg,
        out=np.full(len(cases), np.nan),
        where=idcg > 0
    )
    return cases


def evaluate(data, probabilities=None, criteria=None):
    cases = case_scores(
        data, top_five(data, probabilities, criteria)
    )

    result = {
        name: float(cases[name].mean())
        for name in [
            "Recall@5", "Precision@5",
            "NDCG@5", "Precision@5_all"
        ]
    }

    result["eligible_user_months"] = int(
        (cases["positive_assets"] > 0).sum()
    )

    if probabilities is None:
        result.update({"LogLoss": None, "ROC-AUC": None})
    else:
        p = np.asarray(probabilities, dtype=float)
        assert len(p) == len(data)
        assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()

        p = np.clip(p, 1e-15, 1 - 1e-15)
        y = data["label"].to_numpy()

        result["LogLoss"] = float(
            -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()
        )
        result["ROC-AUC"] = (
            float(roc_auc_score(y, p))
            if len(np.unique(y)) > 1 else None
        )

    return result

In [10]:
def fit_version(columns, trees=100, leaves=7):
    model = HistGradientBoostingClassifier(
        max_iter=trees,
        max_leaf_nodes=leaves,
        early_stopping=False,
        random_state=seed
    )

    with threadpool_limits(limits=4):
        model.fit(train[columns], train["label"])
        probabilities = model.predict_proba(
            validation[columns]
        )[:, 1]

    return model, probabilities


all_features_model, all_features_probabilities = fit_version(
    feature_names
)

settings = [(100, 7), (100, 15), (200, 7), (200, 15)]
models = {}
predictions = {}
tuning_rows = []

for trees, leaves in settings:
    model, probabilities = fit_version(
        selected_features, trees, leaves
    )

    models[(trees, leaves)] = model
    predictions[(trees, leaves)] = probabilities

    tuning_rows.append({
        "max_iter": trees,
        "max_leaf_nodes": leaves,
        **evaluate(validation, probabilities)
    })

tuning = pd.DataFrame(tuning_rows).sort_values(
    ["Recall@5", "LogLoss", "max_iter", "max_leaf_nodes"],
    ascending=[False, True, True, True]
).reset_index(drop=True)

best_key = (
    int(tuning.iloc[0]["max_iter"]),
    int(tuning.iloc[0]["max_leaf_nodes"])
)

automatic_model = models[best_key]
automatic_probabilities = predictions[best_key]

print(tuning.round(5).to_string(index=False))
print("Выбраны настройки:", best_key)

 max_iter  max_leaf_nodes  Recall@5  Precision@5  NDCG@5  Precision@5_all  eligible_user_months  LogLoss  ROC-AUC
      100               7   0.62273      0.24615 0.52187          0.03947                   481  0.02730  0.93289
      200               7   0.60413      0.24075 0.51156          0.03860                   481  0.02852  0.93224
      100              15   0.60396      0.24116 0.50722          0.03867                   481  0.02773  0.92992
      200              15   0.59176      0.23534 0.50115          0.03773                   481  0.02801  0.92780
Выбраны настройки: (100, 7)


In [11]:
manual_saved = joblib.load(
    project_dir / "outputs/manual_baseline_reviewed_model.joblib"
)

with threadpool_limits(limits=4):
    manual_probabilities = manual_saved["model"].predict_proba(
        validation[manual_saved["features"]]
    )[:, 1]

manual_metrics = evaluate(validation, manual_probabilities)

old_selected = next(
    row for row in baseline["comparison"]
    if row["method_id"] == "boost_selected"
)

for metric in ["Recall@5", "LogLoss", "ROC-AUC"]:
    assert np.isclose(
        manual_metrics[metric],
        old_selected[metric],
        rtol=0, atol=1e-12
    )

old_control = next(
    row for row in baseline["comparison"]
    if row["method_id"] == "boost_5_control"
)

metric_names = [
    "Recall@5", "Precision@5", "NDCG@5",
    "Precision@5_all", "LogLoss", "ROC-AUC"
]

rule_criteria = [
    "user_item_buy_count_3m",
    "item_buy_count_1m"
]

method_specs = {
    "manual_selected": {"probabilities": manual_probabilities},
    "all_17_100_7": {"probabilities": all_features_probabilities},
    "selected_100_7": {"probabilities": predictions[(100, 7)]},
    "automatic_selected": {"probabilities": automatic_probabilities},
    "pair_then_popularity": {"criteria": rule_criteria}
}

comparison_rows = [{
    "method": "manual_5_100_7",
    **{name: old_control[name] for name in metric_names}
}]

comparison_rows += [
    {"method": name, **evaluate(validation, **spec)}
    for name, spec in method_specs.items()
]

comparison = pd.DataFrame(comparison_rows)

print(
    comparison[["method"] + metric_names]
    .round(5).to_string(index=False, na_rep="—")
)

monthly_rows = []

for date, month in validation.groupby("prediction_date"):
    for name, spec in method_specs.items():
        if "criteria" in spec:
            month_spec = {"criteria": spec["criteria"]}
        else:
            month_spec = {
                "probabilities": pd.Series(
                    spec["probabilities"], index=validation.index
                ).reindex(month.index)
            }

        monthly_rows.append({
            "month": str(date.date()),
            "method": name,
            **evaluate(month, **month_spec)
        })

monthly_comparison = pd.DataFrame(monthly_rows)

              method  Recall@5  Precision@5  NDCG@5  Precision@5_all  LogLoss  ROC-AUC
      manual_5_100_7   0.60821      0.24449 0.51982          0.03920  0.02754  0.92953
     manual_selected   0.62047      0.24782 0.52676          0.03973  0.02762  0.92887
        all_17_100_7   0.61265      0.24491 0.52992          0.03927  0.02724  0.93253
      selected_100_7   0.62273      0.24615 0.52187          0.03947  0.02730  0.93289
  automatic_selected   0.62273      0.24615 0.52187          0.03947  0.02730  0.93289
pair_then_popularity   0.61544      0.24532 0.53504          0.03933        —        —


In [12]:
def recall_cases(data, probabilities=None, criteria=None):
    return case_scores(
        data, top_five(data, probabilities, criteria)
    )["Recall@5"]


selected_cases = recall_cases(validation, automatic_probabilities)
eligible = selected_cases.notna()
clients = baseline["client_ids"]

references = {
    "manual_selected": {"probabilities": manual_probabilities},
    "pair_then_popularity": {"criteria": rule_criteria}
}

counts = (
    eligible.groupby(level="customerID").sum()
    .reindex(clients, fill_value=0).to_numpy()
)

delta_sums = np.column_stack([
    (
        selected_cases[eligible]
        - recall_cases(validation, **spec)[eligible]
    )
    .groupby(level="customerID").sum()
    .reindex(clients, fill_value=0).to_numpy()
    for spec in references.values()
])

bootstrap_seed = 20261013
bootstrap_repeats = 5000
rng = np.random.default_rng(bootstrap_seed)

draw_differences = np.full(
    (bootstrap_repeats, len(references)), np.nan
)

for iteration in range(bootstrap_repeats):
    draw = rng.integers(0, len(clients), size=len(clients))
    denominator = counts[draw].sum()

    if denominator > 0:
        draw_differences[iteration] = (
            delta_sums[draw].sum(axis=0) / denominator
        )

bootstrap_rows = []

for column, name in enumerate(references):
    values = draw_differences[:, column]
    valid = np.isfinite(values)

    low, high = (
        np.quantile(values[valid], [0.025, 0.975])
        if valid.any() else (np.nan, np.nan)
    )

    bootstrap_rows.append({
        "reference": name,
        "difference": float(delta_sums[:, column].sum() / counts.sum()),
        "ci_low": float(low),
        "ci_high": float(high),
        "contains_zero": bool(low <= 0 <= high) if valid.any() else None,
        "empty_draws": int((~valid).sum())
    })

bootstrap = pd.DataFrame(bootstrap_rows)
print(bootstrap.round(5).to_string(index=False))

           reference  difference   ci_low  ci_high  contains_zero  empty_draws
     manual_selected     0.00226 -0.01744  0.02140           True            0
pair_then_popularity     0.00729 -0.01144  0.02658           True            0


In [13]:
scores = comparison.set_index("method")

delta = (
    scores.loc["automatic_selected", "Recall@5"]
    - scores.loc["manual_selected", "Recall@5"]
)

conclusions = [
    f"Создано {len(auto_names)} новых кандидатов.",
    f"Автоматически выбрано {len(selected_new)} новых признаков.",
    f"Итоговый набор: {len(selected_features)} признаков.",
    f"Выбраны настройки: {best_key[0]}/{best_key[1]}.",
    f"Recall ручной модели: {scores.loc['manual_selected', 'Recall@5']:.4f}.",
    f"Recall автоматической модели: {scores.loc['automatic_selected', 'Recall@5']:.4f}.",
    f"Разница Recall: {delta:+.4f}.",
    "Человек задаёт рамки генерации и правило отбора; программа создаёт и выбирает варианты.",
    "Сходные признаки могут заменять друг друга; отбор по LogLoss не гарантирует улучшения Recall.",
    "Validation использован для выбора настроек. Test не оценён."
]

for row in bootstrap_rows:
    note = (
        "устойчивое преимущество не подтверждено"
        if row["contains_zero"]
        else "интервал не включает ноль, но нужна независимая проверка"
    )
    conclusions.append(
        f"Сравнение с {row['reference']}: "
        f"[{row['ci_low']:+.4f}, {row['ci_high']:+.4f}]; {note}."
    )

print("\n".join(conclusions))

Создано 12 новых кандидатов.
Автоматически выбрано 5 новых признаков.
Итоговый набор: 10 признаков.
Выбраны настройки: 100/7.
Recall ручной модели: 0.6205.
Recall автоматической модели: 0.6227.
Разница Recall: +0.0023.
Человек задаёт рамки генерации и правило отбора; программа создаёт и выбирает варианты.
Сходные признаки могут заменять друг друга; отбор по LogLoss не гарантирует улучшения Recall.
Validation использован для выбора настроек. Test не оценён.
Сравнение с manual_selected: [-0.0174, +0.0214]; устойчивое преимущество не подтверждено.
Сравнение с pair_then_popularity: [-0.0114, +0.0266]; устойчивое преимущество не подтверждено.


In [14]:
def clean_json(value):
    if isinstance(value, dict):
        return {key: clean_json(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json(item) for item in value]
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.bool_):
        return bool(value)
    return value


record = {
    "status": "automatic_extension_validation_frozen",
    "candidate_plan": feature_plan,
    "manual_features": manual_features,
    "selected_new_features": selected_new,
    "features": selected_features,
    "parameters": {
        "max_iter": best_key[0],
        "max_leaf_nodes": best_key[1],
        "learning_rate": float(automatic_model.learning_rate),
        "early_stopping": False,
        "random_state": seed
    },
    "feature_selection": {
        "method": "permutation_importance",
        "scoring": "neg_log_loss",
        "inner_train": ["2020-04-01", "2021-06-01"],
        "inner_check": ["2021-07-01", "2021-12-01"],
        "max_new_features": max_new_features,
        "positive_mean_required": True,
        "tie_rule": "importance descending, feature name ascending",
        "repeats": permutation_repeats,
        "seed": seed,
        "importance": importance_table.to_dict("records")
    },
    "comparison": comparison.to_dict("records"),
    "monthly_results": monthly_comparison.to_dict("records"),
    "tuning": tuning.to_dict("records"),
    "bootstrap": {
        "seed": bootstrap_seed,
        "repeats": bootstrap_repeats,
        "unit": "client",
        "replacement": True,
        "client_pool_size": len(clients),
        "draw_size": len(clients),
        "paired_draws": True,
        "averaging": "eligible user-months with client multiplicity",
        "empty_draws": "excluded from percentiles and counted",
        "diagnostic_only": True,
        "results": bootstrap_rows
    },
    "experiment_budget": {
        "boosting_fits_per_full_run": 6,
        "new_candidates": len(auto_names),
        "selection_model_fits": 1,
        "all_features_control_fits": 1,
        "tuning_fits": 4,
        "permutation_prediction_calls": 1 + len(feature_names) * permutation_repeats,
        "inner_quality_prediction_calls": 1,
        "earlier_experiments": None,
        "equal_total_cost_with_manual_baseline": False
    },
    "conclusions": conclusions,
    "versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__
    },
    "test_evaluated": False
}

for key in [
    "train_months", "validation_months", "test_months",
    "client_ids", "asset_ids", "cohort_random_state",
    "cohort_selection_window", "metric_definitions",
    "tie_break", "observation", "limits",
    "catalog_coverage", "transactions_sha256",
    "history_protocol", "label_definition", "selection_rule"
]:
    record[key] = baseline[key]

assert hashlib.sha256(
    (project_dir / "work/data/FAR-Trans/transactions.csv").read_bytes()
).hexdigest() == record["transactions_sha256"]

record = clean_json(record)
output_dir = project_dir / "outputs"

joblib.dump(
    {
        "model": automatic_model,
        "features": selected_features,
        "metadata": record
    },
    output_dir / "automatic_features_model.joblib"
)

(output_dir / "automatic_features_results.json").write_text(
    json.dumps(record, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8"
)

print("Модель и результаты сохранены.")

Модель и результаты сохранены.


### Что мы получили

Список инструкций для будущего расчёта: название признака, группа и длина окна. Пока здесь нет ни прогнозов, ни оценки качества.

Следующим отдельным шагом научим код считать эти варианты по истории сделок. Используем только события строго до даты прогноза. Train, validation, клиенты и каталог останутся теми же, что в ручном baseline; test остаётся закрытым.

**Для объяснения на защите:** автоматический перебор описаний — начало генератора; полезность признаков ещё предстоит проверить. Окно в 30 дней отличается от календарного месяца.
